# DS 340W — Stomach TCM Syndrome Classification

This notebook is a **course reimplementation** of the workflow described in Gong et al. (2023), *A syndrome differentiation model of TCM based on multi-label deep forest using biomedical text mining*.

The parent paper describes this sequence:

1. remove cases with missing symptoms or syndromes;
2. combine repeated single-label rows into multi-label clinical cases;
3. standardize symptom descriptions;
4. convert symptoms to TF-IDF features;
5. rank/select features with PCC-MLRF;
6. train a multi-label classifier and evaluate it with multi-label metrics.

For DS 340W, we also keep a **70% training / 20% test / 10% final unseen validation** split.

> Important: the authors' public GitHub repository contains the kidney and stomach spreadsheets, but not the official ML-PRDF source code. The PCC-MLRF stage below is therefore a transparent paper-inspired course reimplementation, and the classifier is a reproducible multi-output Random Forest baseline. The final 10% validation data is created and saved but is **not used for model fitting, feature selection, tuning, or test evaluation** in this notebook.

## 1. Setup

The notebook downloads the stomach dataset directly from the parent paper's public GitHub repository, so you do not need to manually upload `ds340wstomach.xlsx` to Colab.

In [ ]:
!pip -q install pandas numpy scikit-learn openpyxl

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    hamming_loss, precision_score, recall_score, f1_score, accuracy_score,
    coverage_error, label_ranking_loss, label_ranking_average_precision_score,
)

RANDOM_STATE = 0

## 2. Load the public stomach dataset

The parent paper cites `web333panda/TCM-Dataset`. The raw stomach spreadsheet has `ID`, `症候`, `症状`, and `编号`. A single clinical case can occur on multiple rows because one case can have multiple syndrome labels.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/web333panda/TCM-Dataset/"
    "main/%E8%83%83%E9%83%A8%E7%96%BE%E7%97%85%E6%95%B0%E6%8D%AE.xlsx"
)
stomach = pd.read_excel(DATA_URL, engine="openpyxl")
print("Raw rows:", len(stomach))
print("Unique clinical cases:", stomach["编号"].nunique())
print("Unique raw syndrome labels:", stomach["症候"].nunique())
print("Columns:", stomach.columns.tolist())
stomach.head()

### Note about the paper's reported statistics

The public raw spreadsheet is not identical to the paper's final post-standardization table. The paper reports **436 stomach cases, 323 symptom features, and 49 syndrome labels after data integration and standardization**. The public raw spreadsheet contains more cases/labels before those undocumented standardization/filtering details are applied. This notebook reproduces the published workflow using the public raw data, but it does not claim to exactly reconstruct the authors' private SymMap synonym dictionary or exact final table.

## 3. Basic cleaning before splitting

Only unusable rows with missing case ID, syndrome, or symptoms are removed. This is deterministic cleaning, not learned preprocessing.

In [ ]:
clean_rows = stomach.dropna(subset=["编号", "症候", "症状"]).copy()
clean_rows["症候"] = clean_rows["症候"].astype(str).str.strip()
clean_rows["症状"] = clean_rows["症状"].astype(str).str.strip()
clean_rows = clean_rows[(clean_rows["症候"] != "") & (clean_rows["症状"] != "")].copy()
print("Rows after basic cleaning:", len(clean_rows))
print("Unique cases after basic cleaning:", clean_rows["编号"].nunique())

## 4. Create the 70% / 20% / 10% split by clinical case

Split on `编号`, **not individual spreadsheet rows**, so the same clinical case cannot leak across partitions.

In [ ]:
case_ids = pd.Series(clean_rows["编号"].unique())
development_ids, validation_ids = train_test_split(case_ids, test_size=0.10, random_state=RANDOM_STATE, shuffle=True)
training_ids, test_ids = train_test_split(development_ids, test_size=2/9, random_state=RANDOM_STATE, shuffle=True)
training_rows = clean_rows[clean_rows["编号"].isin(training_ids)].copy()
test_rows = clean_rows[clean_rows["编号"].isin(test_ids)].copy()
validation_rows = clean_rows[clean_rows["编号"].isin(validation_ids)].copy()
print("TRAIN:", training_rows["编号"].nunique(), "cases /", len(training_rows), "rows")
print("TEST:", test_rows["编号"].nunique(), "cases /", len(test_rows), "rows")
print("VALIDATION:", validation_rows["编号"].nunique(), "cases /", len(validation_rows), "rows")

In [ ]:
train_case_ids = set(training_rows["编号"])
test_case_ids = set(test_rows["编号"])
validation_case_ids = set(validation_rows["编号"])
assert train_case_ids.isdisjoint(test_case_ids)
assert train_case_ids.isdisjoint(validation_case_ids)
assert test_case_ids.isdisjoint(validation_case_ids)
print("Leakage check: PASS")

## 5. Save the three assignment datasets

These files satisfy the course split requirement. After the validation file is created, the rest of this notebook does not use it.

In [ ]:
training_rows.to_excel("stomach_training.xlsx", index=False)
test_rows.to_excel("stomach_test.xlsx", index=False)
validation_rows.to_excel("stomach_validation_UNSEEN.xlsx", index=False)
print("Saved all three split files.")

In [ ]:
del validation_rows
del validation_ids
del validation_case_ids
print("Validation removed from active workflow.")

## 6. Parent-paper-style multi-label case construction

Repeated single-label rows are grouped by `编号`; symptoms are normalized and all syndrome labels are collected for each case.

In [ ]:
def normalize_symptom_text(text):
    text = str(text).strip().replace("，", "、").replace(",", "、")
    text = re.sub(r"\s+", "", text)
    return "、".join(dict.fromkeys([x for x in text.split("、") if x]))

def combine_to_multilabel_cases(rows):
    work = rows.dropna(subset=["编号", "症候", "症状"]).copy()
    work["症候"] = work["症候"].astype(str).str.strip()
    work["症状"] = work["症状"].map(normalize_symptom_text)
    return work.groupby("编号", as_index=False).agg(
        symptoms=("症状", lambda s: normalize_symptom_text("、".join(s))),
        syndromes=("症候", lambda s: sorted(set(s))),
    )

training_cases = combine_to_multilabel_cases(training_rows)
test_cases = combine_to_multilabel_cases(test_rows)
print("Training multi-label cases:", len(training_cases))
print("Test multi-label cases:", len(test_cases))
training_cases.head()

## 7. TF-IDF — fit on training only

The vectorizer treats each Chinese symptom phrase separated by `、` as one token. Vocabulary and IDF are learned only from training data.

In [ ]:
vectorizer = TfidfVectorizer(tokenizer=lambda s: [x for x in s.split("、") if x], token_pattern=None, lowercase=False, min_df=1)
X_train = vectorizer.fit_transform(training_cases["symptoms"])
X_test = vectorizer.transform(test_cases["symptoms"])
label_binarizer = MultiLabelBinarizer()
Y_train = label_binarizer.fit_transform(training_cases["syndromes"])
Y_test = label_binarizer.transform(test_cases["syndromes"])
print("Training feature matrix:", X_train.shape)
print("Test feature matrix:", X_test.shape)
print("Number of syndrome labels:", Y_train.shape[1])

## 8. PCC-MLRF-style feature ranking

The paper describes Pearson-correlation-based sample similarity, traversal through training samples, same-label neighbors (Hits), different-label neighbors (Misses), and feature-weight ranking. The official author code is not in the cited public dataset repository, so this is a paper-inspired course implementation.

In [ ]:
def pearson_similarity_matrix(X, eps=1e-8):
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X, dtype=float)
    centered = X - X.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(centered, axis=1, keepdims=True)
    normalized = centered / np.maximum(norms, eps)
    rho = np.clip(normalized @ normalized.T, -1.0, 1.0 - eps)
    return 1.0 / np.maximum(1.0 - rho, eps)

def rank_features_pcc_mlrf(X, Y, k=5):
    Xd = X.toarray() if hasattr(X, "toarray") else np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=int)
    n, f = Xd.shape
    similarities = pearson_similarity_matrix(Xd)
    np.fill_diagonal(similarities, -np.inf)
    weights = np.zeros(f)
    feature_range = Xd.max(axis=0) - Xd.min(axis=0)
    feature_range[feature_range == 0] = 1.0
    for t in range(n):
        same = np.all(Y == Y[t], axis=1)
        same[t] = False
        different = ~np.all(Y == Y[t], axis=1)
        hit_candidates = np.flatnonzero(same)
        miss_candidates = np.flatnonzero(different)
        if len(hit_candidates):
            hits = hit_candidates[np.argsort(similarities[t, hit_candidates])[::-1][:k]]
            s = np.maximum(similarities[t, hits], 0)
            if s.sum() > 0:
                d = np.abs(Xd[hits] - Xd[t]) / feature_range
                weights -= (s[:, None] * d).sum(axis=0) / s.sum()
        if len(miss_candidates):
            misses = miss_candidates[np.argsort(similarities[t, miss_candidates])[::-1][:k]]
            s = np.maximum(similarities[t, misses], 0)
            if s.sum() > 0:
                d = np.abs(Xd[misses] - Xd[t]) / feature_range
                weights += (s[:, None] * d).sum(axis=0) / s.sum()
    weights /= n
    return np.argsort(weights)[::-1], weights

ranking, feature_weights = rank_features_pcc_mlrf(X_train, Y_train, k=5)
feature_names = np.asarray(vectorizer.get_feature_names_out())
pd.DataFrame({"feature": feature_names[ranking[:20]], "weight": feature_weights[ranking[:20]]})

## 9. Select top-ranked features

For the first implementation, keep the top 100 features or all features if there are fewer than 100.

In [ ]:
TOP_N_FEATURES = min(100, X_train.shape[1])
selected_indices = ranking[:TOP_N_FEATURES]
X_train_selected = X_train[:, selected_indices]
X_test_selected = X_test[:, selected_indices]
print("Original TF-IDF features:", X_train.shape[1])
print("Selected features:", X_train_selected.shape[1])

## 10. Train the multi-label baseline

This uses one Random Forest per syndrome label. It is a reproducible baseline for verifying the complete parent-paper-style pipeline before attempting a specialized MLDF cascade.

In [ ]:
model = MultiOutputClassifier(RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"), n_jobs=-1)
model.fit(X_train_selected, Y_train)
Y_pred = np.asarray(model.predict(X_test_selected))
print("Model trained.")

## 11. Test-set evaluation

Lower is better for Hamming loss, one-error, coverage error, and ranking loss. Higher is better for average precision and F1.

In [ ]:
def positive_class_scores(model, X):
    outputs = model.predict_proba(X)
    cols = []
    for estimator, probabilities in zip(model.estimators_, outputs):
        classes = list(estimator.classes_)
        cols.append(probabilities[:, classes.index(1)] if 1 in classes else np.zeros(probabilities.shape[0]))
    return np.column_stack(cols)

def one_error(y_true, y_score):
    top = np.argmax(y_score, axis=1)
    return float(np.mean([y_true[i, j] == 0 for i, j in enumerate(top)]))

Y_score = positive_class_scores(model, X_test_selected)
metrics = {
    "Hamming loss": hamming_loss(Y_test, Y_pred),
    "Micro precision": precision_score(Y_test, Y_pred, average="micro", zero_division=0),
    "Micro recall": recall_score(Y_test, Y_pred, average="micro", zero_division=0),
    "Micro F1": f1_score(Y_test, Y_pred, average="micro", zero_division=0),
    "Macro F1": f1_score(Y_test, Y_pred, average="macro", zero_division=0),
    "Exact-match accuracy": accuracy_score(Y_test, Y_pred),
    "One-error": one_error(Y_test, Y_score),
    "Coverage error": coverage_error(Y_test, Y_score),
    "Ranking loss": label_ranking_loss(Y_test, Y_score),
    "Average precision": label_ranking_average_precision_score(Y_test, Y_score),
}
pd.DataFrame({"metric": metrics.keys(), "value": metrics.values()})

## 12. Inspect test predictions

Use only for debugging on the test set.

In [ ]:
predicted_label_sets = label_binarizer.inverse_transform(Y_pred)
true_label_sets = label_binarizer.inverse_transform(Y_test)
preview = test_cases[["编号", "symptoms"]].copy()
preview["true_syndromes"] = [list(x) for x in true_label_sets]
preview["predicted_syndromes"] = [list(x) for x in predicted_label_sets]
preview.head(10)

## 13. Validation remains untouched

Stop here during development. Do not load the unseen file until you are bug-free and satisfied with training/test results.

In [ ]:
# FINAL VALIDATION ONLY — DO NOT RUN DURING DEVELOPMENT
# validation_rows = pd.read_excel("stomach_validation_UNSEEN.xlsx", engine="openpyxl")
# validation_cases = combine_to_multilabel_cases(validation_rows)
# X_validation = vectorizer.transform(validation_cases["symptoms"])
# Y_validation = label_binarizer.transform(validation_cases["syndromes"])
# X_validation_selected = X_validation[:, selected_indices]
# Y_validation_pred = np.asarray(model.predict(X_validation_selected))
# print("FINAL VALIDATION Hamming loss:", hamming_loss(Y_validation, Y_validation_pred))

## What is and is not reproduced

**Reproduced from the paper's stated workflow:** public stomach data, missing-record removal, multi-label case grouping, symptom-token normalization, TF-IDF, Pearson-based PCC-MLRF-style feature ranking, multi-label classification, and evaluation.

**Not exactly reproducible from the public materials:** the authors' full SymMap synonym dictionary, exact post-standardization 436-case/49-label stomach table, and official ML-PRDF/MLDF source code.